# ⚡ Universal Hardware-Accelerated WebGPU Chrome on Google Colab (Tesla T4)
### پلتفرم جامع ابری برای انواع پروژه‌های ماینینگ اثبات کار (PoW)، ان‌اف‌تی‌های پردازشی، توکن‌ها و وب۳ با شتاب کامل کارت گرافیک

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hosein-ul/colab-webgpu-chrome/blob/main/colab_chrome_webgpu.ipynb)

این پلتفرم به یک پروژه خاص محدود نیست؛ بلکه یک محیط کامل **Google Chrome ابری با شتاب سخت‌افزاری کارت گرافیک NVIDIA Tesla T4 (۱۵ گیگابایت VRAM)** است که می‌توانید هر سایت یا برنامه ماینینگ وب۳، توکن‌های محاسباتی، مینت ان‌اف‌تی‌های نیازمند پردازش سنگین GPU/CPU یا شیدرهای محاسباتی WebGPU را در آن باز کرده، ولت‌های خود (مانند OKX Wallet یا MetaMask) را وصل کرده و با حداکثر توان ماین کنید.

---
### ⚡ انتخاب روش دسترسی:
* **روش اول (موتور مدرن KasmVNC - ۱۰۰٪ تک‌کلیک و بدون اکانت):** استریم فوق‌العاده سبک با فشرده‌سازی داینامیک WebP، بدون کوچک‌ترین لگ در نشانگر موس (Client-side cursor) و بدون نیاز به کپی کردن هیچ کدی.
* **روش دوم (Google Chrome Remote Desktop - استریم ویدئویی ۶۰ فریم):** برای کارهای فوق‌سنگین و طولانی با پروتکل WebRTC و تاخیر ناچیز زیر ۳۰ میلی‌ثانیه.

> ⚠️ **نکته:** مطمئن شوید از منوی بالای صفحه گزینه **Runtime ➔ Change runtime type** روی **T4 GPU** تنظیم شده باشد.

In [ ]:
#@title 🟢 [روش اول] راه‌اندازی ۱۰۰٪ تک‌کلیک با KasmVNC مدرن (Live Real-Time Logs)
#@markdown ### 🌐 تنظیمات آدرس پروژه و رزولوشن صفحه:
#@markdown آدرس سایت پروژه ماینینگ، مینت NFT، توکن PoW یا دپ مورد نظر خود را وارد کنید:
TARGET_URL = "https://google.com" #@param {type:"string"}
RESOLUTION = "1280x720" #@param ["1280x720", "1366x768", "1600x900", "1920x1080"]

import os, sys, time, subprocess, re
from IPython.display import display, HTML

def run_step(step_num, title, cmd, show_output=False):
    print(f"\n▶ [{step_num}/6] {title}...", flush=True)
    t0 = time.time()
    if show_output:
        proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in iter(proc.stdout.readline, ''):
            l = line.strip()
            if l: print(f"  │ {l[:110]}", flush=True)
        proc.wait()
        code = proc.returncode
    else:
        code = subprocess.run(cmd, shell=True).returncode
    dt = time.time() - t0
    status = "✔ انجام شد" if code == 0 else f"✖ خطا (کد {code})"
    print(f"  └─ {status} در {dt:.1f} ثانیه", flush=True)
    return code

# 1. کرنل‌نودهای GPU و درایور ولکان
run_step(1, "پیکربندی سخت‌افزاری هسته لینوکس و کرنل‌نودهای NVIDIA T4", """
os_mknod() {
  mkdir -p /dev/dri /etc/vulkan/icd.d /usr/share/vulkan/icd.d
  mknod -m 666 /dev/dri/card0 c 226 0 2>/dev/null || true
  mknod -m 666 /dev/dri/renderD128 c 226 128 2>/dev/null || true
  mknod -m 666 /dev/nvidia-modeset c 195 254 2>/dev/null || true
  chmod -R 666 /dev/dri /dev/nvidia* 2>/dev/null || true
  echo '{\n  "file_format_version": "1.0.0",\n  "ICD": {\n    "library_path": "libGLX_nvidia.so.0",\n    "api_version": "1.3.275"\n  }\n}' > /etc/vulkan/icd.d/nvidia_icd.json
  cp /etc/vulkan/icd.d/nvidia_icd.json /usr/share/vulkan/icd.d/nvidia_icd.json
  echo '/usr/lib64-nvidia' > /etc/ld.so.conf.d/nvidia.conf
  ldconfig
}
os_mknod
""")

# 2. کاربر اختصاصی و مجوزها
run_step(2, "ساخت کاربر امن colab و تنظیم گروه‌های دسترسی GPU", """
mkdir -p /home/colab
id -u colab >/dev/null 2>&1 || useradd -m -s /bin/bash colab >/dev/null 2>&1
echo 'colab:123456' | chpasswd
usermod -aG sudo,video,render colab >/dev/null 2>&1 || true
echo 'colab ALL=(ALL) NOPASSWD:ALL' > /etc/sudoers.d/colab
""")

# 3. نصب Google Chrome Stable
if not os.path.exists("/usr/bin/google-chrome"):
    run_step(3, "دانلود و نصب رسمی آخرین نسخه Google Chrome Stable", """
    wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
    dpkg -i /tmp/chrome.deb >/dev/null 2>&1 || apt-get install -fy -qq >/dev/null 2>&1
    rm -f /tmp/chrome.deb
    """)
else:
    print("\n▶ [3/6] بررسی مرورگر: Google Chrome از قبل نصب است.", flush=True)

# 4. نصب KasmVNC 1.5.0
res = subprocess.run("lsb_release -cs", shell=True, capture_output=True, text=True)
distro = res.stdout.strip() if res.stdout.strip() in ["jammy", "noble", "focal"] else "jammy"
kasm_deb = f"/tmp/kasmvnc_{distro}.deb"

if not os.path.exists("/usr/bin/vncserver") or not os.path.exists("/usr/share/kasmvnc"):
    run_step(4, f"دانلود و راه‌اندازی موتور مدرن KasmVNC 1.5.0 ({distro}) با فشرده‌سازی WebP", f"""
    DEBIAN_FRONTEND=noninteractive apt-get update -qq
    DEBIAN_FRONTEND=noninteractive apt-get install -y -qq xfce4 xfce4-terminal desktop-base xauth ssl-cert >/dev/null 2>&1
    wget -q -O {kasm_deb} https://github.com/kasmtech/KasmVNC/releases/download/v1.5.0/kasmvncserver_{distro}_1.5.0_amd64.deb
    dpkg -i {kasm_deb} >/dev/null 2>&1 || apt-get install -fy -qq >/dev/null 2>&1
    rm -f {kasm_deb}
    adduser colab ssl-cert >/dev/null 2>&1 || true
    """)
else:
    print("\n▶ [4/6] بررسی KasmVNC: پکیج‌ها از قبل نصب هستند.", flush=True)

# تنظیمات xstartup برای کروم با پرچم‌های شتاب کامل WebGPU
os.makedirs("/home/colab/.vnc", exist_ok=True)
xstartup = f"""#!/bin/bash
unset SESSION_MANAGER
unset DBUS_SESSION_BUS_ADDRESS
startxfce4 &
sleep 2
export VK_ICD_FILENAMES=/etc/vulkan/icd.d/nvidia_icd.json
google-chrome-stable \\
  --no-sandbox \\
  --disable-gpu-sandbox \\
  --enable-unsafe-webgpu \\
  --enable-features=Vulkan,DefaultANGLEVulkan,VulkanFromANGLE,WebGPUService \\
  --use-vulkan=native \\
  --use-angle=vulkan \\
  --ignore-gpu-blocklist \\
  --disable-dev-shm-usage \\
  --enable-gpu-rasterization \\
  --enable-zero-copy \\
  --start-maximized \\
  "{TARGET_URL}" &
"""
with open("/home/colab/.vnc/xstartup", "w") as f: f.write(xstartup)
subprocess.run("chmod +x /home/colab/.vnc/xstartup", shell=True)
subprocess.run("echo '123456' | kasmvncpasswd -u colab -wo >/dev/null 2>&1 || true", shell=True)
subprocess.run("chown -R colab:colab /home/colab", shell=True)

# 5. تونل Cloudflare
if not os.path.exists("/usr/local/bin/cloudflared"):
    run_step(5, "دانلود باینری تونل امن و پرسرعت Cloudflare", """
    wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    chmod +x /usr/local/bin/cloudflared
    """)
else:
    print("\n▶ [5/6] بررسی تونل: Cloudflared آماده است.", flush=True)

# 6. اجرای سرویس‌ها
print("\n▶ [6/6] استارت سرور KasmVNC و ایجاد لینک ارتباطی زنده...", flush=True)
subprocess.run("su - colab -c 'vncserver -kill :1' >/dev/null 2>&1 || true", shell=True)
subprocess.run("pkill -9 -f 'cloudflared' >/dev/null 2>&1 || true", shell=True)
time.sleep(1)

subprocess.run(f"su - colab -c 'vncserver :1 -geometry {RESOLUTION} -depth 24 -disableBasicAuth'", shell=True)
time.sleep(2)

log_file = "/tmp/cloudflared.log"
if os.path.exists(log_file): os.remove(log_file)
subprocess.Popen(["cloudflared", "tunnel", "--url", "https://localhost:8443", "--no-tls-verify", "--logfile", log_file])

tunnel_url = None
for _ in range(25):
    time.sleep(1)
    if os.path.exists(log_file):
        with open(log_file) as f:
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
            if m:
                tunnel_url = m.group(0)
                break

if tunnel_url:
    print("\n" + "═"*65)
    print("🎉 محیط ابری با موفقیت لود شد! (شتاب سخت‌افزاری Tesla T4 فعال است)")
    print(f"🎯 آدرس لودشده در مرورگر: {TARGET_URL}")
    print("═"*65)
    display(HTML(f'''
        <div style="background: linear-gradient(135deg, #0f2027 0%, #203a43 50%, #2c5364 100%); padding: 24px; border-radius: 12px; color: #fff; font-family: sans-serif; box-shadow: 0 4px 20px rgba(0,0,0,0.3); text-align: center; margin: 15px 0;">
            <h2 style="margin: 0 0 10px 0; color: #00e676;">⚡ دسکتاپ KasmVNC آماده اتصال است!</h2>
            <p style="font-size: 15px; margin-bottom: 20px; opacity: 0.95;">
                مرورگر کروم با فشرده‌سازی WebP و بدون هیچ لگ موس آماده شد. برای ورود روی دکمه زیر کلیک کنید:
            </p>
            <a href="{tunnel_url}" target="_blank" style="padding: 14px 34px; background: #00e676; color: #000; font-weight: bold; font-size: 18px; text-decoration: none; border-radius: 8px; display: inline-block; box-shadow: 0 4px 15px rgba(0,230,118,0.4);">
                👉 ورود به مرورگر ابری (KasmVNC)
            </a>
            <p style="margin-top: 15px; font-size: 13px; opacity: 0.8;">
                لینک مستقیم: <a href="{tunnel_url}" target="_blank" style="color: #64ffda;">{tunnel_url}</a>
            </p>
        </div>
    '''))
else:
    print("❌ خطا در ایجاد آدرس تونل. لطفاً سلول را دوباره اجرا نمایید.")


In [ ]:
#@title ⚡ [روش دوم] راه‌اندازی با Google Chrome Remote Desktop (Live Real-Time Logs)
#@markdown ### 📋 راهنمای اتصال ریموت دسکتاپ گوگل (تاخیر زیر ۳۰ میلی‌ثانیه و ۶۰ فریم):
#@markdown 1. در یک تب جدید به **[remotedesktop.google.com/headless](https://remotedesktop.google.com/headless)** بروید.
#@markdown 2. دکمه‌های **Begin** ➔ **Next** ➔ **Authorize** را بزنید و کد لینوکس دبیان را کپی کنید.
#@markdown 3. کد را در فیلد `AUTH_COMMAND` زیر پیست کرده و دکمه **Play** را بزنید:

AUTH_COMMAND = "" #@param {type:"string"}
PIN = 123456 #@param {type:"integer"}
TARGET_URL = "https://google.com" #@param {type:"string"}

import os, sys, time, subprocess, re
from IPython.display import display, HTML

raw_auth = AUTH_COMMAND.strip()
if not raw_auth:
    print("⚠️ فیلد AUTH_COMMAND خالی است!")
    print("👈 دریافت کد لینوکس: https://remotedesktop.google.com/headless")
    raw_auth = input("دستور احراز هویت را وارد کنید: ").strip()

if not raw_auth:
    raise ValueError("دستور احراز هویت الزامی است.")

def run_step(step_num, title, cmd):
    print(f"\n▶ [{step_num}/5] {title}...", flush=True)
    t0 = time.time()
    code = subprocess.run(cmd, shell=True).returncode
    dt = time.time() - t0
    status = "✔ انجام شد" if code == 0 else f"✖ خطا (کد {code})"
    print(f"  └─ {status} در {dt:.1f} ثانیه", flush=True)
    return code

# 1. درایورهای گرافیک T4
run_step(1, "پیکربندی کرنل‌نودها و درایورهای Vulkan تسلا T4", """
mkdir -p /dev/dri /etc/vulkan/icd.d /usr/share/vulkan/icd.d
mknod -m 666 /dev/dri/card0 c 226 0 2>/dev/null || true
mknod -m 666 /dev/dri/renderD128 c 226 128 2>/dev/null || true
mknod -m 666 /dev/nvidia-modeset c 195 254 2>/dev/null || true
chmod -R 666 /dev/dri /dev/nvidia* 2>/dev/null || true
echo '{\n  "file_format_version": "1.0.0",\n  "ICD": {\n    "library_path": "libGLX_nvidia.so.0",\n    "api_version": "1.3.275"\n  }\n}' > /etc/vulkan/icd.d/nvidia_icd.json
cp /etc/vulkan/icd.d/nvidia_icd.json /usr/share/vulkan/icd.d/nvidia_icd.json
echo '/usr/lib64-nvidia' > /etc/ld.so.conf.d/nvidia.conf
ldconfig
""")

# 2. کاربر اختصاصی
run_step(2, "ساخت کاربر امن colab و تنظیم گروه‌های کاربری", """
mkdir -p /home/colab
id -u colab >/dev/null 2>&1 || useradd -m -s /bin/bash colab >/dev/null 2>&1
echo 'colab:123456' | chpasswd
usermod -aG sudo,video,render colab >/dev/null 2>&1 || true
echo 'colab ALL=(ALL) NOPASSWD:ALL' > /etc/sudoers.d/colab
""")

# 3. پکیج‌های XFCE و Chrome Remote Desktop
run_step(3, "نصب پکیج‌های XFCE4 و Chrome Remote Desktop", """
DEBIAN_FRONTEND=noninteractive apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq xfce4 desktop-base xfce4-terminal xscreensaver >/dev/null 2>&1
systemctl disable lightdm.service >/dev/null 2>&1 || true
if [ ! -f /opt/google/chrome-remote-desktop/start-host ]; then
  wget -q -O /tmp/crd.deb https://dl.google.com/linux/direct/chrome-remote-desktop_current_amd64.deb
  dpkg -i /tmp/crd.deb >/dev/null 2>&1 || apt-get install -fy -qq >/dev/null 2>&1
  rm -f /tmp/crd.deb
fi
usermod -aG chrome-remote-desktop colab >/dev/null 2>&1 || true
echo 'exec /etc/X11/Xsession /usr/bin/xfce4-session' > /etc/chrome-remote-desktop-session
echo 'exec /etc/X11/Xsession /usr/bin/xfce4-session' > /home/colab/.chrome-remote-desktop-session
""")

# 4. کروم و اسکریپت WebGPU
run_step(4, "نصب Chrome و تنظیم اسکریپت اجرای خودکار WebGPU", f"""
if [ ! -f /usr/bin/google-chrome ]; then
  wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
  dpkg -i /tmp/chrome.deb >/dev/null 2>&1 || apt-get install -fy -qq >/dev/null 2>&1
  rm -f /tmp/chrome.deb
fi
mkdir -p /home/colab/.config/autostart /home/colab/Desktop
cat << 'EOF' > /home/colab/launch_chrome.sh
#!/bin/bash
export VK_ICD_FILENAMES=/etc/vulkan/icd.d/nvidia_icd.json
google-chrome-stable \\
  --no-sandbox \\
  --disable-gpu-sandbox \\
  --enable-unsafe-webgpu \\
  --enable-features=Vulkan,DefaultANGLEVulkan,VulkanFromANGLE,WebGPUService \\
  --use-vulkan=native \\
  --use-angle=vulkan \\
  --ignore-gpu-blocklist \\
  --disable-dev-shm-usage \\
  --enable-gpu-rasterization \\
  --enable-zero-copy \\
  --start-maximized \\
  "{TARGET_URL}" &
EOF
chmod +x /home/colab/launch_chrome.sh
cat << 'EOF' > /home/colab/.config/autostart/webgpu-chrome.desktop
[Desktop Entry]
Version=1.0
Type=Application
Name=WebGPU Google Chrome
Exec=/home/colab/launch_chrome.sh
Icon=google-chrome
Terminal=false
EOF
cp /home/colab/.config/autostart/webgpu-chrome.desktop /home/colab/Desktop/
chmod +x /home/colab/Desktop/webgpu-chrome.desktop
chown -R colab:colab /home/colab && chmod +x /home/colab/.chrome-remote-desktop-session
""")

# 5. استارت هاست با رانر تمیز
print("\n▶ [5/5] راه‌اندازی هاست ریموت دسکتاپ گوگل با اکانت شما...", flush=True)
clean_cmd = re.sub(r'^DISPLAY=\s*', '', raw_auth)
clean_cmd = re.sub(r'--pin=\S+', '', clean_cmd).strip()

runner_script = f"""#!/bin/bash
printf '{PIN}\n{PIN}\n' | {clean_cmd}
"""
with open("/tmp/start_crd.sh", "w") as f: f.write(runner_script)
subprocess.run("chmod +x /tmp/start_crd.sh && chown colab:colab /tmp/start_crd.sh", shell=True)

res = subprocess.run("su - colab -c /tmp/start_crd.sh", shell=True, capture_output=True, text=True)
time.sleep(3)
check = subprocess.run("ps aux | grep -v grep | grep chrome-remote-desktop", shell=True, capture_output=True, text=True)

print("\n" + "═"*65)
if "chrome-remote-desktop" in check.stdout:
    print("🎉 سرویس Google Chrome Remote Desktop با موفقیت استارت شد!")
    print(f"🔑 پین اتصال: {PIN}")
    print("═"*65)
    display(HTML(f'''
        <div style="background: linear-gradient(135deg, #0d47a1 0%, #1976d2 100%); padding: 24px; border-radius: 12px; color: #fff; font-family: sans-serif; box-shadow: 0 4px 20px rgba(0,0,0,0.3); text-align: center; margin: 15px 0;">
            <h2 style="margin: 0 0 10px 0; color: #64ffda;">🚀 دسکتاپ ابری ۶۰ فریم آماده اتصال است!</h2>
            <p style="font-size: 15px; margin-bottom: 20px; opacity: 0.95;">
                دستگاه <b>colab-t4</b> در پنل ریموت گوگل ثبت شد. روی دکمه زیر کلیک کنید و پین <b>{PIN}</b> را وارد کنید:
            </p>
            <a href="https://remotedesktop.google.com/access" target="_blank" style="padding: 14px 34px; background: #00e676; color: #000; font-weight: bold; font-size: 18px; text-decoration: none; border-radius: 8px; display: inline-block; box-shadow: 0 4px 15px rgba(0,230,118,0.4);">
                👉 ورود به Chrome Remote Desktop
            </a>
            <p style="margin-top: 15px; font-size: 13px; opacity: 0.8;">
                آدرس مستقیم: <a href="https://remotedesktop.google.com/access" target="_blank" style="color: #64ffda;">https://remotedesktop.google.com/access</a>
            </p>
        </div>
    '''))
else:
    print("⚠️ پیام خروجی:")
    print(res.stdout)
    if res.stderr: print("خطاها:", res.stderr)
    print("═"*65)
    print("نکته: در صورت خطای منقضی شدن توکن، کد تازه‌ای از remotedesktop.google.com/headless دریافت کنید.")


In [ ]:
#@title 📊 مانیتورینگ زنده کارت گرافیک و توان پردازشی تسلا T4 (Universal Web3 GPU Monitor)
#@markdown این سلول وضعیت زنده درصد لود پردازشی GPU، حافظه VRAM، توان مصرفی وات، دما و پروسه‌های فعال را برای هر پروژه ماینینگ نمایش می‌دهد:

import time, subprocess
from IPython.display import clear_output

print("Starting Universal GPU Monitor (press Stop button in notebook to halt)...")
try:
    while True:
        res = subprocess.run([
            "nvidia-smi",
            "--query-gpu=name,utilization.gpu,utilization.memory,memory.used,memory.total,temperature.gpu,power.draw",
            "--format=csv,noheader,nounits"
        ], capture_output=True, text=True)
        parts = [p.strip() for p in res.stdout.strip().split(",")]
        if len(parts) >= 7:
            name, gpu_util, mem_util, mem_used, mem_total, temp, power = parts
            clear_output(wait=True)
            print("="*65)
            print(f"🚀 UNIVERSAL WEBGPU MINING & COMPUTE MONITOR — {name}")
            print("="*65)
            print(f"  ⚡ GPU Compute Utilization : {gpu_util}%")
            print(f"  🧠 VRAM Memory Used        : {mem_used} MiB / {mem_total} MiB ({mem_util}%)")
            print(f"  🌡️  GPU Temperature         : {temp} °C")
            print(f"  🔋 Power Consumption       : {power} W")
            print("="*65)
            print("  💡 Press the Stop button (■) in notebook to halt the monitor.")
        time.sleep(1)
except KeyboardInterrupt:
    print("Monitor stopped.")
